# Isogeny Results Viewer

This notebook is a lightweight browser for the single canonical results file
`isogeny_results.txt` placed in the repository root.

It is intended for the public repository, where the workflow is:
- keep one verified `isogeny_results.txt` at the top level,
- open this notebook,
- choose a level `N` from the dropdown,
- inspect the stored formulas and sample fibers.

For bielliptic levels, the stored section contains additional quotient data.
Besides the four Weierstrass coefficient maps, the viewer may display:
- the elliptic quotient curve,
- the quotient map `x_E(t,\alpha_t)`,
- a chosen lift primitive variable on `X_0(N)`,
- a quadratic relation for that primitive variable over the quotient coordinates,
- a linear relation recovering the other coordinate,
- and sample quadratic fibers above the displayed quotient points.

The labels `Lift Primitive Variable on X_0(N)` and `Lift Other Variable on X_0(N)`
record which coordinate was chosen during the reconstruction step. They are not
canonical and may differ from one bielliptic level to another.

For very large formulas, especially in some bielliptic cases, the notebook may skip
full MathJax rendering and show the raw LaTeX code instead, to keep the viewer responsive.


In [ ]:
from __future__ import annotations

from pathlib import Path
import html
import re

import ipywidgets as widgets
from IPython.display import HTML, Math, display

RESULTS_PATH = Path("isogeny_results.txt")
SECTION_RE = re.compile(r"^=== N = (\d+) ===$")

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Expected isogeny_results.txt in the repository root. "
        "Place the canonical public results file next to this notebook."
    )


def parse_section(lines: list[str]) -> dict:
    fields = []
    samples = []
    current_sample = None

    for raw in lines:
        if not raw.strip():
            continue

        line = raw.rstrip("\n")
        stripped = line.strip()
        if stripped and set(stripped) == {"-"}:
            continue

        if stripped.startswith("Sample Multiple n="):
            current_sample = {"header": stripped, "fibers": []}
            samples.append(current_sample)
            continue

        if line.startswith("  Fiber ") and current_sample is not None:
            current_sample["fibers"].append(stripped)
            continue

        if ":" in stripped:
            key, value = stripped.split(":", 1)
            fields.append((key.strip(), value.strip()))
        else:
            fields.append((stripped, ""))

    return {"fields": fields, "samples": samples}


def parse_results_file(path: Path) -> dict[int, dict]:
    sections = {}
    current_n = None
    current_lines = []

    for raw in path.read_text().splitlines():
        match = SECTION_RE.match(raw.strip())
        if match:
            if current_n is not None:
                sections[current_n] = parse_section(current_lines)
            current_n = int(match.group(1))
            current_lines = []
            continue
        if current_n is not None:
            current_lines.append(raw)

    if current_n is not None:
        sections[current_n] = parse_section(current_lines)

    return sections


def latexify_label(label: str) -> str:
    return label.replace("alpha_t", r"\alpha_t")


MAX_MATH_RENDER_CHARS = 8000
SKIP_MATH_RENDER_LEVELS = {131}


def should_render_math(level: int, latex_text: str) -> bool:
    return level not in SKIP_MATH_RENDER_LEVELS and len(latex_text) <= MAX_MATH_RENDER_CHARS


def render_math_with_codebox(lhs: str, rhs: str | None = None, title: str | None = None, render_math: bool = True) -> None:
    full_latex = lhs if rhs is None else (f"{lhs} = {rhs}" if lhs else rhs)

    box_layout = widgets.Layout(
        border="1px solid #d0d0d0",
        padding="15px",
        margin="0 0 16px 0",
        border_radius="8px",
        width="98%",
    )

    container = widgets.Output(layout=box_layout)
    with container:
        if title:
            display(
                HTML(
                    f"<div style='font-weight:600; margin-bottom:8px; font-family:sans-serif;'>"
                    f"{html.escape(title)}</div>"
                )
            )
        if render_math:
            display(Math(full_latex))
        else:
            if lhs:
                display(Math(lhs))
            display(
                HTML(
                    "<div style='color:#7a5c00; background:#fff8e1; border:1px solid #ecd58b; "
                    "border-radius:6px; padding:10px 12px; margin-bottom:10px; font-family:sans-serif;'>"
                    "Math rendering was skipped for this formula to keep the viewer responsive."
                    "</div>"
                )
            )
        escaped = html.escape(full_latex)
        display(
            HTML(
                "<div style='margin-top:12px; border-top:1px dashed #eee; padding-top:10px;'>"
                "<div style='font-size:0.85em; color:#666; margin-bottom:5px; font-weight:600; font-family:sans-serif;'>"
                "Raw LaTeX code (click to select):"
                "</div>"
                f"<textarea readonly onclick='this.select();' style='width:100%; height:54px; padding:8px; "
                "font-family:Courier New, monospace; font-size:0.9em; background-color:#f8f9fa; "
                "border:1px solid #ccc; border-radius:4px; resize:vertical; box-sizing:border-box;'>"
                f"{escaped}</textarea></div>"
            )
        )
    display(container)


def render_text_block(label: str, value: str) -> None:
    display(
        HTML(
            "<div style='border:1px solid #e0e0e0; border-radius:8px; padding:12px 15px; margin:0 0 12px 0;'>"
            f"<div style='font-weight:600; font-family:sans-serif; margin-bottom:6px;'>{html.escape(label)}</div>"
            f"<div style='font-family:Courier New, monospace; white-space:pre-wrap; word-break:break-word;'>{html.escape(value)}</div>"
            "</div>"
        )
    )


def render_samples(samples: list[dict]) -> None:
    if not samples:
        return
    display(HTML("<h3 style='font-family:sans-serif; margin-top:24px;'>Sample Fibers</h3>"))
    for sample in samples:
        display(HTML(f"<h4 style='font-family:sans-serif; margin:12px 0 8px 0;'>{html.escape(sample['header'])}</h4>"))
        for fiber in sample["fibers"]:
            render_text_block("Fiber", fiber)


def render_section(level: int, section: dict, output: widgets.Output) -> None:
    with output:
        output.clear_output()
        display(HTML(f"<h2 style='font-family:sans-serif;'>N = {level}</h2>"))
        display(HTML(f"<div style='color:#555; margin-bottom:16px; font-family:sans-serif;'>Source file: {html.escape(str(RESULTS_PATH))}</div>"))

        for key, value in section["fields"]:
            if key == "Family Type":
                display(HTML(f"<div style='font-family:sans-serif; margin-bottom:16px;'><b>Family Type:</b> {html.escape(value)}</div>"))
            elif key in {"Base Curve", "Quotient Curve Equation"}:
                render_math_with_codebox("", value, title=key, render_math=should_render_math(level, value))
            elif key in {"Lift Quadratic Relation over Q(x_n,y_n)", "Lift Linear Relation over Q(x_n,y_n)"}:
                render_math_with_codebox("", value, title=key, render_math=should_render_math(level, value))
            elif key == "Quotient Map x_E(t,alpha_t)":
                render_math_with_codebox(r"x_E(t,\alpha_t)", value, render_math=should_render_math(level, value))
            elif key.startswith("a_") or key.startswith("a'_") or key.startswith(r"\Delta") or key.startswith("Delta("):
                render_math_with_codebox(latexify_label(key), value, render_math=should_render_math(level, value))
            else:
                render_text_block(key, value)

        render_samples(section["samples"])


sections = parse_results_file(RESULTS_PATH)
levels = sorted(sections)

level_dropdown = widgets.Dropdown(
    options=levels,
    value=levels[0],
    description="Level N:",
    layout=widgets.Layout(width="30%"),
)

output = widgets.Output()


def render_current(*_args):
    level = int(level_dropdown.value)
    render_section(level, sections[level], output)


level_dropdown.observe(render_current, names="value")

viewer = widgets.VBox(
    [
        widgets.HTML("<h2 style='font-family:sans-serif;'>Browse Precomputed Isogeny Families</h2>"),
        level_dropdown,
        output,
    ]
)

display(viewer)
render_current()


## Notes

- This notebook is intentionally read-only.
- The public repository should expose exactly one canonical results file named `isogeny_results.txt`.
- If the verified results file is regenerated later, replace that file in place; the notebook does not need to change.
